In [1]:
%pip install numpy
%pip install matplotlib
%pip install pandas
%pip install scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys 

from pathlib import Path

from sklearn.model_selection import ParameterGrid, TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge, LinearRegression, RidgeCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer
from sklearn.multioutput import MultiOutputRegressor

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
class MismoParameterSelector:
    """
    Selects the structural hyperparameters (d, na, s) for the MISMO strategy
    by training a MISMO model for each candidate combination and evaluating it
    via rollout NMSE on a held-out validation split.

    Mirrors the interface of ParameterSelector so both selectors can be used
    interchangeably inside MismoTransformer.

    Parameters
    ----------
    u_train : np.ndarray, shape (N, n_in)
        Training input series.
    y_train : np.ndarray, shape (N, 2)
        Training output series.
    u_val : np.ndarray, shape (M, n_in)
        Validation input series (used for rollout evaluation).
    y_val : np.ndarray, shape (M, 2)
        Ground-truth validation outputs.
    val_ratio : float
        Fraction of u_train / y_train to hold out when u_val / y_val are not
        supplied explicitly.  Ignored when u_val and y_val are provided.
    random_state : int
        Seed passed to any stochastic model.
    """

    def __init__(
        self,
        u_train: np.ndarray,
        y_train: np.ndarray,
        u_val: np.ndarray  = None,
        y_val: np.ndarray  = None,
        val_ratio: float   = 0.30,
        random_state: int  = 42,
    ) -> None:
        self.u_train      = np.asarray(u_train, dtype=float)
        self.y_train      = np.asarray(y_train, dtype=float)
        self.val_ratio    = float(val_ratio)
        self.random_state = int(random_state)

        if self.u_train.ndim != 2 or self.y_train.ndim != 2:
            raise ValueError("u_train and y_train must be 2-D arrays.")
        if self.u_train.shape[0] != self.y_train.shape[0]:
            raise ValueError("u_train and y_train must have the same number of rows.")
        if self.y_train.shape[1] != 2:
            raise ValueError("MismoParameterSelector expects exactly 2 output channels.")
        if not (0.0 < self.val_ratio < 1.0):
            raise ValueError("val_ratio must be in (0, 1).")

        # If explicit val arrays are given use them directly; otherwise carve
        # a chronological hold-out from the training data.
        if u_val is not None and y_val is not None:
            self._u_tr_fit  = self.u_train
            self._y_tr_fit  = self.y_train
            self._u_val     = np.asarray(u_val, dtype=float)
            self._y_val     = np.asarray(y_val, dtype=float)
        else:
            split = int(len(self.u_train) * (1.0 - self.val_ratio))
            self._u_tr_fit  = self.u_train[:split]
            self._y_tr_fit  = self.y_train[:split]
            self._u_val     = self.u_train[split:]
            self._y_val     = self.y_train[split:]

    # ------------------------------------------------------------------
    # shared static helper  (same as ParameterSelector)
    # ------------------------------------------------------------------

    @staticmethod
    def cross_correlation(
        y: np.ndarray, u: np.ndarray, max_lag: int = 40
    ):
        """Delegates to the module-level cross_correlation function."""
        return cross_correlation(y=y, u=u, max_lag=max_lag)

    # ------------------------------------------------------------------
    # delay estimation  (identical logic to ParameterSelector)
    # ------------------------------------------------------------------

    def estimate_delay_candidates(
        self,
        max_lag: int = 40,
        max_d:   int = 10,
        baseline_stop:        int = 4,
        around_median_radius: int = 2,
        around_min_radius:    int = 1,
    ):
        """
        Estimates candidate values for d by cross-correlating every
        (input j, output i) pair and pooling the positive-lag peaks.

        Returns
        -------
        d_candidates : list[int]
        peak_lags    : dict[(input_idx, output_idx) -> lag]
        """
        nu = self.u_train.shape[1]
        ny = self.y_train.shape[1]
        peak_lags = {}

        for i in range(nu):
            for j in range(ny):
                lags, corr = self.cross_correlation(
                    self.y_train[:, j], self.u_train[:, i], max_lag=max_lag
                )
                pos_mask = lags >= 0
                if np.any(pos_mask):
                    peak = int(lags[pos_mask][np.argmax(np.abs(corr[pos_mask]))])
                else:
                    peak = 0
                peak_lags[(i, j)] = peak

        values = np.array(list(peak_lags.values()), dtype=int)
        d_min  = int(values.min()) if values.size else 0
        d_med  = int(np.median(values)) if values.size else 0

        baseline    = set(range(0, max(1, baseline_stop + 1)))
        around_med  = set(range(max(0, d_med - around_median_radius),
                                d_med + around_median_radius + 2))
        around_min  = set(range(max(0, d_min - around_min_radius),
                                d_min + around_min_radius + 2))
        d_candidates = sorted(
            d for d in (baseline | around_med | around_min)
            if 0 <= d <= max_d
        )
        if not d_candidates:
            d_candidates = [0, 1, 2]

        return d_candidates, peak_lags

    # ------------------------------------------------------------------
    # core evaluation  (MISMO-specific, replaces evaluate_config)
    # ------------------------------------------------------------------

    def evaluate_config(
        self,
        d:          int,
        na:         int,
        s:          int,
        model=None,
    ):
        """
        Train a MISMO model on the fit split, run a block rollout on the
        validation split, and return a result dict.

        Parameters
        ----------
        d, na, s : int
            Structural hyperparameters to evaluate.
        model : sklearn estimator or None
            Regression model passed to train_mismo.  Defaults to
            RidgeCV(alphas=ALPHAS).

        Returns
        -------
        dict with keys d, na, s, nmse_val, rmse_val, n_train, n_val
        or None if there are not enough samples for this configuration.
        """
        # need at least s samples after the warm-up horizon in the fit split
        min_train = (d + na) + s + 1
        if len(self._u_tr_fit) < min_train or len(self._u_val) < s:
            return None

        try:
            fitted_model = train_mismo(
                self._u_tr_fit, self._y_tr_fit,
                d=d, na=na, s=s,
                model=model,
            )
        except Exception:
            return None

        y_hat = mismo_rollout(
            model     = fitted_model,
            U_test    = self._u_val,
            d         = d,
            na        = na,
            s         = s,
            y_prefix  = self._y_tr_fit,   # warm-start the output buffer
        )

        return {
            "d":        int(d),
            "na":       int(na),
            "s":        int(s),
            "nmse_val": float(nmse(self._y_val, y_hat)),
            "rmse_val": float(rmse(self._y_val, y_hat)),
            "n_train":  int(len(self._u_tr_fit)),
            "n_val":    int(len(self._u_val)),
        }

    # ------------------------------------------------------------------
    # grid search
    # ------------------------------------------------------------------

    @staticmethod
    def _sort_cols(objective: str):
        options = {
            "nmse_val": ["nmse_val", "rmse_val"],
            "rmse_val": ["rmse_val", "nmse_val"],
        }
        if objective not in options:
            raise ValueError("objective must be 'nmse_val' or 'rmse_val'.")
        return options[objective]

    def search(
        self,
        order_grid: dict,
        model      = None,
        objective:  str  = "nmse_val",
        verbose:    bool = False,
    ) -> pd.DataFrame:
        """
        Exhaustive grid search over (d, na, s).

        Parameters
        ----------
        order_grid : dict
            Keys must include 'd', 'na', 's'.
            Example::

                {
                    'd':  [0, 1, 2],
                    'na': [1, 2, 3, 4, 5],
                    's':  [1, 5, 10, 20, 50, 100, 200, 500],
                }

        model : sklearn estimator or None
            Base regressor passed to train_mismo.  None → RidgeCV.
        objective : str
            Primary sort column: 'nmse_val' (default) or 'rmse_val'.
        verbose : bool
            Print a progress line every 25 evaluations.

        Returns
        -------
        pd.DataFrame  sorted by objective (best first), empty if no valid config.
        """
        sort_cols = self._sort_cols(objective)
        rows      = []

        for i, params in enumerate(ParameterGrid(order_grid)):
            row = self.evaluate_config(
                d  = int(params["d"]),
                na = int(params["na"]),
                s  = int(params["s"]),
                model = model,
            )
            if row is not None:
                rows.append(row)

            if verbose and (i + 1) % 25 == 0:
                best_so_far = min((r[sort_cols[0]] for r in rows), default=float("inf"))
                print(f"[MismoParameterSelector] evals={i+1}  "
                      f"best_{sort_cols[0]}={best_so_far:.4f}")

        if not rows:
            return pd.DataFrame()

        df = pd.DataFrame(rows).sort_values(sort_cols).reset_index(drop=True)

        if verbose:
            print(f"\n___ MISMO Grid (d, na, s) ___")
            print(df.to_string(index=False))

        return df

    # ------------------------------------------------------------------
    # extract best config  (same interface as ParameterSelector)
    # ------------------------------------------------------------------

    @staticmethod
    def best_config(results: pd.DataFrame) -> dict:
        """
        Returns the top row as a plain dict with keys 'd', 'na', 's'.
        Raises ValueError if results is empty.
        """
        if results.empty:
            raise ValueError("results is empty; cannot extract best configuration.")
        row = results.iloc[0]
        return {
            "d":  int(row["d"]),
            "na": int(row["na"]),
            "s":  int(row["s"]),
        }

    # ------------------------------------------------------------------
    # convenience: fit full model on all training data with best params
    # ------------------------------------------------------------------

    def fit_full_model(
        self,
        d:    int,
        na:   int,
        s:    int,
        model = None,
    ):
        """
        Retrain a MISMO model on the *entire* training set (not just the fit
        split) using the best hyperparameters found by search().

        Returns the fitted pipeline returned by train_mismo().
        """
        return train_mismo(
            self.u_train, self.y_train,
            d=d, na=na, s=s,
            model=model,
        )

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISMO PIPELINE  ─  drop-in replacement for the MIMO pipeline
# ─────────────────────────────────────────────────────────────────────────────

class MismoTransformer(BaseEstimator, TransformerMixin):
    """
    Builds the MISMO feature matrix for a block-of-s-steps strategy.

    For each anchor time k the feature vector is:
        X(k) = [Y(k-d), ..., Y(k-d-na)]          ← output lags  (na+1) × 2
                || [U(k+1), ..., U(k+s)]           ← KNOWN future inputs  s × n_in

    The target vector (used only during .fit) is:
        T(k) = [Y(k+1), Y(k+2), ..., Y(k+s)]     ← s × 2  (flattened to s*2)

    Parameters
    ----------
    d, na, s : int or None
        Structural parameters.  If any is None the transformer will call
        ParameterSelector (the same helper used by LaggedTransformer) to
        find the best (d, na) pair during .fit().  's' must always be
        supplied explicitly (it controls the block size).
    u_future : np.ndarray, shape (N, n_in), optional
        The full future-input array that is available at test time.
        Must be set before calling .transform() on test data.
    """

    def __init__(
        self,
        d=None,
        na=None,
        s=10,
        order_grid=None,
        selection_model="ridge",
        selection_model_params=None,
        selection_objective="recursive_nmse",
        val_ratio=0.30,
        random_state=42,
        min_samples=30,
        max_lag=40,
        max_d=10,
        u_future=None,          # set this at test time
    ):
        self.d = d
        self.na = na
        self.s = s
        self.order_grid = order_grid
        self.selection_model = selection_model
        self.selection_model_params = selection_model_params
        self.selection_objective = selection_objective
        self.val_ratio = val_ratio
        self.random_state = random_state
        self.min_samples = min_samples
        self.max_lag = max_lag
        self.max_d = max_d
        self.u_future = u_future   # injected at prediction time

    # ------------------------------------------------------------------
    # helpers
    # ------------------------------------------------------------------

    def _resolve_params(self, data, y):
        """Run ParameterSelector to find best d and na if not supplied."""
        X_df = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        u_tr = X_df[["u1(k)", "u2(k)"]].to_numpy(dtype=float)
        y_tr = np.asarray(y, dtype=float)
        if y_tr.ndim == 1:
            y_tr = y_tr.reshape(-1, 1)

        selector = ParameterSelector(
            u_train=u_tr, y_train=y_tr,
            val_ratio=self.val_ratio, random_state=self.random_state,
        )

        grid = self.order_grid
        if grid is None:
            d_cands, _ = selector.estimate_delay_candidates(
                max_lag=self.max_lag, max_d=self.max_d
            )
            grid = {
                "d":  d_cands,
                "na": list(range(1, 10)),
                "nb": list(range(1, 10)),   # nb is ignored by MISMO but selector needs it
            }

        results = selector.search(
            grid,
            model_name=self.selection_model,
            model_params=self.selection_model_params,
            objective=self.selection_objective,
            min_samples=self.min_samples,
        )
        if results.empty:
            raise RuntimeError("MismoTransformer: parameter selection returned no valid config.")

        best = ParameterSelector.best_config(results)
        return int(best["d"]), int(best["na"])

    # ------------------------------------------------------------------
    # sklearn API
    # ------------------------------------------------------------------

    def fit(self, X, y=None):
        if self.d is None or self.na is None:
            self.d_, self.na_ = self._resolve_params(X, y)
        else:
            self.d_  = int(self.d)
            self.na_ = int(self.na)
        self.s_ = int(self.s)

        # remember the training U and Y for the rollout
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self._u_train = X_df[["u1(k)", "u2(k)"]].to_numpy(dtype=float)

        if "y1(k)" in X_df.columns:
            self._y_train = X_df[["y1(k)", "y2(k)"]].to_numpy(dtype=float)

        return self

    def transform(self, X, y=None):
        """
        Build the MISMO feature matrix.

        During training  : X must be the full training DataFrame (with y columns).
        During test time : X is the test DataFrame (inputs only, no y columns).
                           self.u_future must be set to the test input array.
        """
        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        d, na, s = self.d_, self.na_, self.s_

        is_test = "y1(k)" not in X_df.columns

        if is_test:
            # ── test path ──────────────────────────────────────────────
            # U_all : the full array of inputs available at test time
            U_test = X_df[["u1(k)", "u2(k)"]].to_numpy(dtype=float)
            # We build one feature row per anchor k in the test sequence.
            # Output lags come from a rolling buffer seeded with training y.
            # Future inputs U(k+1..k+s) come from U_test.
            rows = []
            buf_len = d + na + 1
            buf = np.zeros((buf_len, 2))  # output history buffer

            if hasattr(self, "_y_train"):
                take = min(buf_len, len(self._y_train))
                buf[-take:] = self._y_train[-take:]

            for k in range(len(U_test)):
                feat = []
                # output lags  [Y(k-d), ..., Y(k-d-na)]
                for lag in range(d, d + na + 1):
                    idx = buf_len - 1 - lag
                    feat.extend(buf[max(idx, 0)] if idx >= 0 else [0.0, 0.0])
                # future inputs [U(k+1), ..., U(k+s)]
                for h in range(1, s + 1):
                    ui = k + h - 1
                    feat.extend(U_test[ui] if ui < len(U_test) else np.zeros(2))
                rows.append(feat)

            col_names = (
                [f"y1(k-{i})" for i in range(d, d + na + 1)]
                + [f"y2(k-{i})" for i in range(d, d + na + 1)]
                + [f"u1(k+{h})" for h in range(1, s + 1)]
                + [f"u2(k+{h})" for h in range(1, s + 1)]
            )
            return pd.DataFrame(rows, columns=col_names)

        else:
            # ── training path ──────────────────────────────────────────
            Y = X_df[["y1(k)", "y2(k)"]].to_numpy(dtype=float)
            U = X_df[["u1(k)", "u2(k)"]].to_numpy(dtype=float)
            N = len(Y)
            rows = []
            for k in range(d + na, N - s):
                feat = []
                for lag in range(d, d + na + 1):
                    feat.extend(Y[k - lag])
                for h in range(1, s + 1):
                    feat.extend(U[k + h])
                rows.append(feat)

            col_names = (
                [f"y1(k-{i})" for i in range(d, d + na + 1)]
                + [f"y2(k-{i})" for i in range(d, d + na + 1)]
                + [f"u1(k+{h})" for h in range(1, s + 1)]
                + [f"u2(k+{h})" for h in range(1, s + 1)]
            )
            return pd.DataFrame(rows, columns=col_names)


# ─────────────────────────────────────────────────────────────────────────────
# MismoTargetBuilder  (helper used only in .fit, not a pipeline step)
# ─────────────────────────────────────────────────────────────────────────────

def build_mismo_targets(X_df, Y_df, d, na, s):
    """
    Returns T of shape (n_samples, s*2):
        T(k) = [y1(k+1), y2(k+1), y1(k+2), y2(k+2), ..., y1(k+s), y2(k+s)]
    aligned with the rows produced by MismoTransformer.transform(train data).
    """
    Y = np.asarray(Y_df[["y1(k+1)", "y2(k+1)"]].values if hasattr(Y_df, "columns")
                   else Y_df, dtype=float)
    # Y_df rows correspond to k=0..N-1 with Y_df[k] = y(k+1)
    # The transformer anchors at k = d+na .. N-s-1
    # So target for anchor k  is  [y(k+1)..y(k+s)]  = Y_df rows k..k+s-1
    
    # Rebuild from the raw Y matrix instead (simpler indexing)
    Y_raw = np.asarray(Y_df, dtype=float)   # shape (N-1, 2)  where row k = y(k+1)
    N_full = len(Y_raw) + 1                  # original series length

    rows = []
    for k in range(d + na, N_full - s - 1):
        tgt = []
        for h in range(1, s + 1):
            tgt.extend(Y_raw[k + h - 1])    # Y_df[k+h-1] = y(k+h)
        rows.append(tgt)
    return np.array(rows, dtype=float)


# ─────────────────────────────────────────────────────────────────────────────
# MISMO rollout  (replaces multiple_steps_ahead_pipeline)
# ─────────────────────────────────────────────────────────────────────────────

def mismo_rollout_pipeline(pipeline, U_test_df, y_init=None):
    """
    Autoregressive MISMO rollout using a trained Pipeline whose first step
    is a MismoTransformer.

    At each anchor k the pipeline predicts a block of s steps in one call.
    Predicted outputs are fed back into the transformer's output buffer for
    the next block.

    Parameters
    ----------
    pipeline   : fitted sklearn Pipeline with a MismoTransformer first step
    U_test_df  : DataFrame with columns ['u1(k)', 'u2(k)'], length N_test
    y_init     : array-like (M, 2), last M rows of training outputs used to
                 warm-start the output buffer (optional but recommended)

    Returns
    -------
    Y_hat : np.ndarray, shape (N_test, 2)
    """
    trans   = pipeline.named_steps["transformer"]
    imputer = pipeline.named_steps["imputer"]
    scaler  = pipeline.named_steps["scaler"]
    model   = pipeline.named_steps["model"]

    d, na, s = trans.d_, trans.na_, trans.s_
    buf_len = d + na + 1

    # initialise output buffer
    buf = np.zeros((buf_len, 2))
    if y_init is not None:
        take = min(buf_len, len(y_init))
        buf[-take:] = np.asarray(y_init)[-take:]

    U_test = U_test_df[["u1(k)", "u2(k)"]].to_numpy(dtype=float)
    N_test = len(U_test)
    Y_hat  = np.zeros((N_test, 2))

    k = 0
    while k < N_test:
        block = min(s, N_test - k)

        # ── build one feature row for anchor k ───────────────────────
        feat = []
        # output lags
        for lag in range(d, d + na + 1):
            idx = buf_len - 1 - lag
            feat.extend(buf[idx] if idx >= 0 else [0.0, 0.0])
        # future inputs  U(k+1) .. U(k+s)   (pad with zeros at the tail)
        for h in range(1, s + 1):
            ui = k + h - 1
            feat.extend(U_test[ui] if ui < N_test else np.zeros(2))

        feat_arr = np.array(feat, dtype=float).reshape(1, -1)

        # ── run through imputer + scaler + model ─────────────────────
        feat_arr = imputer.transform(feat_arr)
        feat_arr = scaler.transform(feat_arr)

        pred_flat  = model.predict(feat_arr)[0]          # shape (s*2,)
        pred_block = pred_flat.reshape(s, 2)             # shape (s, 2)

        # write the valid part of the block
        Y_hat[k : k + block] = pred_block[:block]

        # update output buffer with predictions
        for step in range(block):
            buf = np.roll(buf, -1, axis=0)
            buf[-1] = Y_hat[k + step]

        k += block

    return Y_hat